In [ ]:
from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.1.200")

import time

servos = [servo for servo in range(11, 71, 10)]

# Joshua's configuration - MC-servo-motor
def fix_servos(angle=70):
    """Sets the servos to a specific angle.
    More positive angle makes car taller (moves arms down).
    """
    angle = max(min(80, angle), -70)
    got.turn_servo_angle(11, -angle, 1000) # front left
    got.turn_servo_angle(31, angle, 1000) # back left
    got.turn_servo_angle(41, -angle, 1000) # back right
    got.turn_servo_angle(61, angle, 1000) # front right

def straight(speed=20):
    """Moves UGOT forward (positive speed) or backward (negative speed)"""
    speed = max(min(80, speed), -80)
    got.turn_motor_speed(11, -speed) # front left
    got.turn_motor_speed(31, -speed) # back left
    got.turn_motor_speed(41, speed) # back right
    got.turn_motor_speed(61, speed) # front right

def move_turn(move_speed=20, turn_speed=20):
    """Move forward/backward while turning left or right.

    Positive turn_speed turns left, negative turn_speed turns right.
    """
    move_speed = max(min(80, move_speed), -80)
    turn_speed = max(min(80, turn_speed), -80)

    left_speed = move_speed - turn_speed
    right_speed = move_speed + turn_speed

    got.turn_motor_speed(11, -left_speed) # front left
    got.turn_motor_speed(31, -left_speed) # back left
    got.turn_motor_speed(41, right_speed) # back right
    got.turn_motor_speed(61, right_speed) # front right

def stop():
    got.stop_all_servos()

# fix_servos(60)
# time.sleep(1)
# straight(10)
# time.sleep(1)
# move_turn(30, -30)
# time.sleep(1)
# stop()

192.168.1.200:50051


In [67]:
def follow_line(speed):
    got.load_models(["line_recognition"])
    got.set_track_recognition_line(0)

    no_line_count = 0
    while True:
        try:
            line_info = got.get_single_track_total_info()
            # print(line_info)
            
            offset = line_info[0]
            line_type = line_info[1]
            if line_type == 0:
                no_line_count += 1
                if no_line_count > 4:
                    stop()
                    break
            else:
                no_line_count = 0
            rot = int(offset*0.2)
            move_turn(speed, rot)
        except:
            stop()
            break

fix_servos(80)
time.sleep(1)
follow_line(20)

Tested: can use built-in commands without DIY model. Just attach mechanical arm and still select transforming vehicle as the model. Using mecanum wheels (why?)

In [ ]:
from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.1.200")
got.load_models(["line_recognition"])
got.set_track_recognition_line(0)
import time

got.transform_set_chassis_height(6)
got.mechanical_joint_control(0, 80, 50, 500)

while True:
    try:
        line_info = got.get_single_track_total_info()
        offset = line_info[0]
        rotation = int(offset * 0.3)
        if rotation < 0:
            turn = 3 # right
        else:
            turn = 2 # left
        got.transform_move_turn(0, 20, turn, abs(rotation))
    except KeyboardInterrupt:
        got.transform_stop()
        break


192.168.1.200:50051
